# NB12 진짜 블록 해부 — 제네시스 블록 80바이트를 손으로 풀기

『법률가를 위한 블록체인 — 기술과 경제학으로 읽기』의 장별 노트북입니다. 본문의 코드와 같습니다.

- 모든 코드는 원리를 눈으로 보기 위한 **장난감 구현**입니다. 실제 자산·실제 개인키·실제 거래에 쓰지 마십시오(실사용 금지).
- 외부 API를 부르지 않습니다. 네트워크 없이 실행됩니다. 동봉 스냅샷 데이터가 필요한 셀은 그 데이터를 셀 안에 적어 두었습니다.
- 표준 라이브러리만 씁니다(Python 3.13에서 실행·확인). 설치 셀은 없습니다.
- 출력은 엮은이가 실행한 결과를 그대로 담아 두었습니다. 다시 실행하면 같은 값이 나와야 합니다(난수는 시드를 고정했습니다).

In [ ]:
# NB12-1: 제네시스 블록 헤더 80바이트를 여섯 필드로 자른다 (동봉 스냅샷 파일, 없으면 같은 바이트의 상수)
import struct, hashlib, datetime, os

HEADER_HEX = ("0100000000000000000000000000000000000000000000000000000000000000000000003ba3edfd7a7b12b27ac72c3e67768f617fc81bc3888a51323a9fb8aa4b1e5e4a29ab5f49ffff001d1dac2b7c")   # data/genesis_header.hex 와 같은 바이트

def load_hex(path, fallback):
    try:
        with open(path) as f: return f.read().strip()
    except OSError:
        return fallback

header = bytes.fromhex(load_hex("data/genesis_header.hex", HEADER_HEX))
print("헤더 길이:", len(header), "바이트")
assert header.hex() == HEADER_HEX, "동봉 파일과 상수가 다르다"

fields = [("버전(nVersion)", 0, 4), ("이전 블록 해시(hashPrevBlock)", 4, 32), ("머클 루트(hashMerkleRoot)", 36, 32),
          ("시각(nTime)", 68, 4), ("목표(nBits)", 72, 4), ("논스(nNonce)", 76, 4)]
for name, off, size in fields:
    raw = header[off:off + size]
    print(f"[{off:>2}:{off+size:>2}] {name:<28} {raw.hex()}")

nVersion = struct.unpack("<I", header[0:4])[0]          # "<" = 리틀엔디안
hashPrev = header[4:36][::-1].hex()                     # 해시는 뒤집어서 표시
merkle   = header[36:68][::-1].hex()
nTime    = struct.unpack("<I", header[68:72])[0]
nBits    = struct.unpack("<I", header[72:76])[0]
nNonce   = struct.unpack("<I", header[76:80])[0]
print()
print("nVersion =", nVersion)
print("hashPrevBlock =", hashPrev, "(제네시스라 0 — 가리킬 부모가 없다)")
print("hashMerkleRoot =", merkle)
print("nTime =", nTime, "=", datetime.datetime.fromtimestamp(nTime, datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"))
print(f"nBits = 0x{nBits:08x}  ({nBits})")
print("nNonce =", nNonce, f"(0x{nNonce:08x})")
print("Core 소스의 CreateGenesisBlock(1231006505, 2083236893, 0x1d00ffff, 1, 50*COIN) 과 일치하는가:",
      (nTime, nNonce, nBits, nVersion) == (1231006505, 2083236893, 0x1d00ffff, 1))

헤더 길이: 80 바이트
[ 0: 4] 버전(nVersion)                 01000000
[ 4:36] 이전 블록 해시(hashPrevBlock)      0000000000000000000000000000000000000000000000000000000000000000
[36:68] 머클 루트(hashMerkleRoot)        3ba3edfd7a7b12b27ac72c3e67768f617fc81bc3888a51323a9fb8aa4b1e5e4a
[68:72] 시각(nTime)                    29ab5f49
[72:76] 목표(nBits)                    ffff001d
[76:80] 논스(nNonce)                   1dac2b7c

nVersion = 1
hashPrevBlock = 0000000000000000000000000000000000000000000000000000000000000000 (제네시스라 0 — 가리킬 부모가 없다)
hashMerkleRoot = 4a5e1e4baab89f3a32518a88c31bc87f618f76673e2cc77ab2127b7afdeda33b
nTime = 1231006505 = 2009-01-03 18:15:05 UTC
nBits = 0x1d00ffff  (486604799)
nNonce = 2083236893 (0x7c2bac1d)
Core 소스의 CreateGenesisBlock(1231006505, 2083236893, 0x1d00ffff, 1, 50*COIN) 과 일치하는가: True


In [ ]:
# NB12-2: 이중 SHA-256 → 블록 해시. 공개값과 일치하는가, 목표값 아래인가, 논스를 바꾸면 어떻게 되는가
def sha256d(b: bytes) -> bytes:
    return hashlib.sha256(hashlib.sha256(b).digest()).digest()

KNOWN = "000000000019d6689c085ae165831e934ff763ae46a2a6c172b3f1b60a8ce26f"   # Core chainparams.cpp 의 assert 값
block_hash = sha256d(header)[::-1].hex()
print("계산한 블록 해시:", block_hash)
print("공개된 블록 해시:", KNOWN)
print("일치:", block_hash == KNOWN)

def nbits_to_target(nbits: int) -> int:
    return (nbits & 0x007FFFFF) * 256 ** ((nbits >> 24) - 3)
target = nbits_to_target(nBits)
h_int = int(block_hash, 16)
print("목표값 T = 0x%064x" % target)
print("해시 ≤ 목표값 (유효한 작업증명):", h_int <= target)
print("해시의 앞 0비트 수:", 256 - h_int.bit_length(), "(목표값이 요구하는 최소는 32)")

# 논스를 1만 바꾸면
tampered = header[:76] + struct.pack("<I", nNonce + 1)
t_hash = sha256d(tampered)[::-1].hex()
print("논스를 2083236894 로 바꾼 헤더의 해시:", t_hash, "→ 목표값 아래?", int(t_hash, 16) <= target)

계산한 블록 해시: 000000000019d6689c085ae165831e934ff763ae46a2a6c172b3f1b60a8ce26f
공개된 블록 해시: 000000000019d6689c085ae165831e934ff763ae46a2a6c172b3f1b60a8ce26f
일치: True
목표값 T = 0x00000000ffff0000000000000000000000000000000000000000000000000000
해시 ≤ 목표값 (유효한 작업증명): True
해시의 앞 0비트 수: 43 (목표값이 요구하는 최소는 32)
논스를 2083236894 로 바꾼 헤더의 해시: 9b227a4a5daa0cbae6874144bc5d7797d0513e320aceadeb3b06304971a41b1c → 목표값 아래? False


In [ ]:
# NB12-3: 블록 전체 285바이트 — 거래 수, 코인베이스 거래의 입력·출력, 그 안의 문장, txid == 머클 루트
BLOCK_HEX = HEADER_HEX + ("0101000000010000000000000000000000000000000000000000000000000000000000000000ffffffff4d04ffff001d0104455468652054696d65732030332f4a616e2f32303039204368616e63656c6c6f72206f6e206272696e6b206f66207365636f6e64206261696c6f757420666f722062616e6b73ffffffff0100f2052a01000000434104678afdb0fe5548271967f1a67130b7105cd6a828e03909a67962e0ea1f61deb649f6bc3f4cef38c4f35504e51ec112de5c384df7ba0b8d578a4c702b6bf11d5fac00000000")
block = bytes.fromhex(load_hex("data/genesis_block.hex", BLOCK_HEX))
assert block.hex() == BLOCK_HEX and block[:80] == header
print("블록 길이:", len(block), "바이트 | 헤더 80 + 나머지", len(block) - 80)

pos = 80
tx_count = block[pos]; pos += 1                    # 거래 수(가변 길이 정수 — 여기서는 1바이트)
print("거래 수:", tx_count)
tx_start = pos
version = struct.unpack("<I", block[pos:pos+4])[0]; pos += 4
in_count = block[pos]; pos += 1
prev_txid = block[pos:pos+32][::-1].hex(); pos += 32
prev_index = struct.unpack("<I", block[pos:pos+4])[0]; pos += 4
script_len = block[pos]; pos += 1
script_sig = block[pos:pos+script_len]; pos += script_len
sequence = struct.unpack("<I", block[pos:pos+4])[0]; pos += 4
print(f"거래 버전 {version}, 입력 수 {in_count}, 이전 출력 = ({prev_txid[:8]}…, 0x{prev_index:08x}) ← 가리키는 출력 없음 = 코인베이스")
print("입력 스크립트(scriptSig) 길이:", script_len, "바이트")

# scriptSig 를 「푸시」 단위로 읽는다: 앞 바이트가 n(1~75)이면 그 뒤 n 바이트를 스택에 올린다
i, pushes = 0, []
while i < len(script_sig):
    n = script_sig[i]; pushes.append(script_sig[i+1:i+1+n]); i += 1 + n
for k, p in enumerate(pushes):
    print(f"  푸시 {k}: {len(p)}바이트  {p.hex() if len(p) <= 4 else p.hex()[:16] + '…'}")
print("  푸시 0 을 리틀엔디안 정수로:", int.from_bytes(pushes[0], "little"), "(= nBits 486604799 와 같은 수)")
print("  푸시 2 를 ASCII 로:", repr(pushes[2].decode("ascii")))

out_count = block[pos]; pos += 1
value = struct.unpack("<Q", block[pos:pos+8])[0]; pos += 8
spk_len = block[pos]; pos += 1
spk = block[pos:pos+spk_len]; pos += spk_len
locktime = struct.unpack("<I", block[pos:pos+4])[0]; pos += 4
print(f"출력 수 {out_count}, 금액 {value:,} 사토시 = {value/1e8} BTC, 잠금 스크립트 {spk_len}바이트, locktime {locktime}")
print("  잠금 스크립트 = [푸시 65바이트] 공개키(앞 바이트 0x%02x) … [0x%02x = OP_CHECKSIG]" % (spk[1], spk[-1]))
print("  공개키 65바이트 =", spk[1:66].hex())

tx_bytes = block[tx_start:pos]
print("거래 길이:", len(tx_bytes), "바이트 | 블록 끝까지 다 읽었는가:", pos == len(block))
txid = sha256d(tx_bytes)[::-1].hex()
print("txid = 이중 SHA-256(거래) =", txid)
print("헤더의 머클 루트       =", merkle)
print("거래가 하나뿐이면 머클 루트 == txid :", txid == merkle)

블록 길이: 285 바이트 | 헤더 80 + 나머지 205
거래 수: 1
거래 버전 1, 입력 수 1, 이전 출력 = (00000000…, 0xffffffff) ← 가리키는 출력 없음 = 코인베이스
입력 스크립트(scriptSig) 길이: 77 바이트
  푸시 0: 4바이트  ffff001d
  푸시 1: 1바이트  04
  푸시 2: 69바이트  5468652054696d65…
  푸시 0 을 리틀엔디안 정수로: 486604799 (= nBits 486604799 와 같은 수)
  푸시 2 를 ASCII 로: 'The Times 03/Jan/2009 Chancellor on brink of second bailout for banks'
출력 수 1, 금액 5,000,000,000 사토시 = 50.0 BTC, 잠금 스크립트 67바이트, locktime 0
  잠금 스크립트 = [푸시 65바이트] 공개키(앞 바이트 0x04) … [0xac = OP_CHECKSIG]
  공개키 65바이트 = 04678afdb0fe5548271967f1a67130b7105cd6a828e03909a67962e0ea1f61deb649f6bc3f4cef38c4f35504e51ec112de5c384df7ba0b8d578a4c702b6bf11d5f
거래 길이: 204 바이트 | 블록 끝까지 다 읽었는가: True
txid = 이중 SHA-256(거래) = 4a5e1e4baab89f3a32518a88c31bc87f618f76673e2cc77ab2127b7afdeda33b
헤더의 머클 루트       = 4a5e1e4baab89f3a32518a88c31bc87f618f76673e2cc77ab2127b7afdeda33b
거래가 하나뿐이면 머클 루트 == txid : True
